# Instacart Online Grocery Shopping Dataset 2017 — Analysis

This notebook explores the [Instacart Online Grocery Shopping Dataset 2017](https://www.instacart.com/datasets/grocery-shopping-2017), which contains a sample of over 3 million grocery orders from more than 200,000 Instacart users.

All visualizations use **Plotly Express** with the `simple_white` template.

In [ ]:
import zipfile
import pandas as pd
import plotly.express as px

# Set default plotly template
px.defaults.template = "simple_white"


## 1 — Load the Data

In [ ]:
import os

# Unzip orders.csv if needed
if not os.path.exists("orders.csv"):
    with zipfile.ZipFile("orders.csv.zip", "r") as z:
        z.extractall(".")

orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")
aisles = pd.read_csv("aisles.csv")
departments = pd.read_csv("departments.csv")
order_products = pd.read_csv("order_products_train.csv")

print(f"orders:          {orders.shape}")
print(f"order_products:  {order_products.shape}")
print(f"products:        {products.shape}")
print(f"aisles:          {aisles.shape}")
print(f"departments:     {departments.shape}")


In [ ]:
# Merge order_products with product, aisle, and department names
op = (
    order_products
    .merge(products, on="product_id")
    .merge(aisles, on="aisle_id")
    .merge(departments, on="department_id")
)
op.head()


## 2 — Orders by Day of Week

In [ ]:
dow_map = {0: "Saturday", 1: "Sunday", 2: "Monday", 3: "Tuesday",
           4: "Wednesday", 5: "Thursday", 6: "Friday"}
dow_counts = (
    orders["order_dow"]
    .map(dow_map)
    .value_counts()
    .reindex([dow_map[i] for i in range(7)])
    .reset_index()
)
dow_counts.columns = ["Day of Week", "Number of Orders"]

fig = px.bar(
    dow_counts,
    x="Day of Week",
    y="Number of Orders",
    title="Number of Orders by Day of Week",
    text_auto=True,
)
fig.show()


## 3 — Orders by Hour of Day

In [ ]:
hour_counts = (
    orders["order_hour_of_day"]
    .value_counts()
    .sort_index()
    .reset_index()
)
hour_counts.columns = ["Hour of Day", "Number of Orders"]

fig = px.bar(
    hour_counts,
    x="Hour of Day",
    y="Number of Orders",
    title="Number of Orders by Hour of Day",
    text_auto=True,
)
fig.show()


## 4 — Top 20 Most Ordered Products

In [ ]:
top_products = (
    op["product_name"]
    .value_counts()
    .head(20)
    .reset_index()
)
top_products.columns = ["Product", "Number of Orders"]

fig = px.bar(
    top_products,
    x="Number of Orders",
    y="Product",
    orientation="h",
    title="Top 20 Most Ordered Products",
    text_auto=True,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()


## 5 — Top 15 Departments by Number of Ordered Items

In [ ]:
dept_counts = (
    op["department"]
    .value_counts()
    .head(15)
    .reset_index()
)
dept_counts.columns = ["Department", "Number of Items Ordered"]

fig = px.bar(
    dept_counts,
    x="Number of Items Ordered",
    y="Department",
    orientation="h",
    title="Top 15 Departments by Number of Ordered Items",
    text_auto=True,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()


## 6 — Top 20 Aisles by Number of Ordered Items

In [ ]:
aisle_counts = (
    op["aisle"]
    .value_counts()
    .head(20)
    .reset_index()
)
aisle_counts.columns = ["Aisle", "Number of Items Ordered"]

fig = px.bar(
    aisle_counts,
    x="Number of Items Ordered",
    y="Aisle",
    orientation="h",
    title="Top 20 Aisles by Number of Ordered Items",
    text_auto=True,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()


## 7 — Reorder Ratio by Department

In [ ]:
reorder_dept = (
    op.groupby("department")["reordered"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
reorder_dept.columns = ["Department", "Reorder Ratio"]

fig = px.bar(
    reorder_dept,
    x="Reorder Ratio",
    y="Department",
    orientation="h",
    title="Reorder Ratio by Department",
    text_auto=".2f",
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()


## 8 — Distribution of Order Sizes (Items per Order)

In [ ]:
order_sizes = (
    order_products
    .groupby("order_id")
    .size()
    .reset_index(name="num_items")
)

fig = px.histogram(
    order_sizes,
    x="num_items",
    nbins=50,
    title="Distribution of Items per Order",
    labels={"num_items": "Number of Items in Order", "count": "Number of Orders"},
)
fig.show()


## 9 — Distribution of Days Since Prior Order

In [ ]:
days_since = orders["days_since_prior_order"].dropna()

fig = px.histogram(
    days_since,
    nbins=31,
    title="Distribution of Days Since Prior Order",
    labels={"value": "Days Since Prior Order", "count": "Number of Orders"},
)
fig.show()


## 10 — Reorder Ratio by Hour of Day

In [ ]:
# Merge order_products with orders to get hour information
op_hour = order_products.merge(orders[["order_id", "order_hour_of_day"]], on="order_id")
reorder_hour = (
    op_hour.groupby("order_hour_of_day")["reordered"]
    .mean()
    .reset_index()
)
reorder_hour.columns = ["Hour of Day", "Reorder Ratio"]

fig = px.line(
    reorder_hour,
    x="Hour of Day",
    y="Reorder Ratio",
    title="Reorder Ratio by Hour of Day",
    markers=True,
)
fig.show()


## 11 — Top 20 Most Reordered Products

In [ ]:
reorder_products = (
    op[op["reordered"] == 1]
    ["product_name"]
    .value_counts()
    .head(20)
    .reset_index()
)
reorder_products.columns = ["Product", "Times Reordered"]

fig = px.bar(
    reorder_products,
    x="Times Reordered",
    y="Product",
    orientation="h",
    title="Top 20 Most Reordered Products",
    text_auto=True,
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()


## 12 — Average Add-to-Cart Position by Department

In [ ]:
cart_order_dept = (
    op.groupby("department")["add_to_cart_order"]
    .mean()
    .sort_values()
    .reset_index()
)
cart_order_dept.columns = ["Department", "Avg Add-to-Cart Position"]

fig = px.bar(
    cart_order_dept,
    x="Avg Add-to-Cart Position",
    y="Department",
    orientation="h",
    title="Average Add-to-Cart Position by Department",
    text_auto=".1f",
)
fig.update_layout(yaxis=dict(autorange="reversed"))
fig.show()
